# Vietnam Stock Market — ML direction forecast (SMA Project)

End-to-end notebook for the ML component of the Stock Market Analytics project.

**Question:** can lecture-style ML models (Decision Tree / Random Forest), fed only
price-derived features known at week *t*, predict whether a Vietnamese stock will be
higher 8 weeks later?

**Setup (point-in-time, no lookahead):**
- Data: Simplize OHLCV (matched-order volume only), weekly bars (close of every 5th session).
- Universe: 254 tickers (>= 5 years of data, last price 2,000-500,000 VND, median daily volume >= 50k shares).
- Features at week t: returns 4/12/26/52w, weekly RSI14, distance to SMA20/SMA50, distance to 52-week high, volume ratio vs 20w mean.
- Target: 1 if close[t+8] > close[t]. Split by time: train <= 2021, test >= 2022.
- Data files are not committed (course rules) - set SMA_DATA_DIR to a local bars store (see data/README.md).

In [1]:
import os
os.environ.setdefault("SMA_DATA_DIR", "data/bars")
import pandas as pd
from ml_model import build_panel, FEATS, TRAIN_END

panel = build_panel()
print(f"Panel: {len(panel):,} ticker-weeks, {panel['date'].min().date()} -> {panel['date'].max().date()}")
panel.head()

Universe: 254 ma giu lai, 1266 ma loai (thieu du lieu / ngoai LIQ)
Panel: 158,671 ticker-weeks, 2002-04-05 -> 2026-09-18


,date,ret4,ret12,ret26,ret52,rsi14,dist_sma20,dist_sma50,dist_52w_high,vol_ratio,fwd8,ticker,y
0,2011-08-08 07:00:00+07:00,-0.151515,-0.289340,-0.596542,-0.689579,29.981771,-0.270643,-0.569390,-0.845133,0.617092,0.234940,AAA,1
1,2011-08-15 07:00:00+07:00,-0.138365,-0.121795,-0.600583,-0.695556,29.539134,-0.262450,-0.570506,-0.848451,0.371521,0.254287,AAA,1
2,2011-08-22 07:00:00+07:00,0.033113,0.075862,-0.547826,-0.688000,35.985106,-0.136929,-0.501916,-0.827434,1.452332,0.013670,AAA,1
3,2011-08-29 07:00:00+07:00,0.143885,-0.091429,-0.457338,-0.641892,36.965657,-0.096334,-0.479064,-0.824115,1.570660,0.014435,AAA,1
4,2011-09-06 07:00:00+07:00,0.100000,-0.241379,-0.442029,-0.653933,35.976553,-0.101517,-0.473144,-0.829646,1.721316,0.081599,AAA,1


In [2]:
tr = panel[panel["date"] <= TRAIN_END]
te = panel[panel["date"] > TRAIN_END]
print(f"Train (<=2021): {len(tr):,} rows | Test (>=2022): {len(te):,} rows")
print(f"Up-rate 8w - train: {tr['y'].mean():.3f} | test: {te['y'].mean():.3f}  (test period leans bearish)")
Xtr, ytr = tr[FEATS], tr["y"]
Xte, yte = te[FEATS], te["y"]

Train (<=2021): 99,118 rows | Test (>=2022): 59,553 rows
Up-rate 8w - train: 0.513 | test: 0.464  (test period leans bearish)


## Models vs baselines

Baselines matter: in the test period only 46.4% of ticker-weeks rose, so "always predict
down" scores 0.536 accuracy and a coin flip has AUC 0.500. A model only counts if it
beats those numbers out of sample.

In [3]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from ml_model import evaluate

results = []
maj = int(ytr.mean() >= 0.5)
base_pred = np.full(len(yte), maj)
results.append({"model": "Majority (blind)", "acc": round(float((base_pred == yte).mean()), 4),
                "auc": 0.5, "f1": round(float(f1_score(yte, base_pred, zero_division=0)), 4)})
print(f"{'Majority (blind)':28s} acc={results[0]['acc']:.3f}  AUC=0.500  F1={results[0]['f1']:.3f}")
print(f"{'Always-down baseline':28s} acc={(yte == 0).mean():.3f}")

for depth in [3, 5, 10, None]:
    results.append(evaluate(f"DecisionTree depth={depth}",
                            DecisionTreeClassifier(max_depth=depth, random_state=0),
                            Xtr, ytr, Xte, yte))
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0)
results.append(evaluate("RandomForest n=200", rf, Xtr, ytr, Xte, yte))
pd.DataFrame(results)

Majority (blind)             acc=0.464  AUC=0.500  F1=0.633
Always-down baseline         acc=0.536


DecisionTree depth=3         acc=0.486  AUC=0.496  F1=0.503


DecisionTree depth=5         acc=0.482  AUC=0.492  F1=0.506


DecisionTree depth=10        acc=0.501  AUC=0.503  F1=0.466


DecisionTree depth=None      acc=0.504  AUC=0.504  F1=0.489


RandomForest n=200           acc=0.507  AUC=0.516  F1=0.506


,model,acc,auc,f1
0,Majority (blind),0.4636,0.5000,0.6335
1,DecisionTree depth=3,0.4860,0.4957,0.5025
2,DecisionTree depth=5,0.4816,0.4919,0.5062
3,DecisionTree depth=10,0.5011,0.5032,0.4663
4,DecisionTree depth=None,0.5038,0.5044,0.4887
5,RandomForest n=200,0.5072,0.5160,0.5062


In [4]:
imp = pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False)
print("Feature importance (RandomForest):")
for k, v in imp.items():
    print(f"  {k:14s} {v:.3f}")

Feature importance (RandomForest):
  dist_sma50     0.119
  ret52          0.117
  ret26          0.116
  rsi14          0.111
  dist_52w_high  0.109
  dist_sma20     0.109
  ret12          0.108
  vol_ratio      0.107
  ret4           0.104


## Regime variant (already run - see results/sma-ml-prototype-regime.md)

Adding the strategy's own regime definition as features (equal-weight index vs its
40-week SMA: `regime_bull`, `dist_eqw_sma40`) made things **worse**, not better:
RandomForest AUC fell to **0.482** (below a coin flip), Decision Trees dropped to
0.469-0.498. Split by regime, the model is below 0.5 in both bull weeks (AUC 0.469)
and bear weeks (0.497) - so the weak base result is not an artefact of a bearish
test period. Reproduce with `python ml_model_regime.py`.

## Conclusion (reported as-is)

Price-only features do **not** predict the 8-week direction on Vietnamese data out
of sample: the best model (RandomForest, AUC 0.516 in the standalone run) barely
matches a coin flip and loses to the always-down baseline on accuracy, and feature
importances are flat. This negative result is the finding. The trading system in
this project therefore stays rule-based (Strategy v1.2, see STRATEGY.md and
results/backtest_summary.md) and does not use ML forecasts as signals.